<a href="https://colab.research.google.com/github/Se886372/Starbucks-Order-Interest-Over-Time/blob/main/Starbucks_Timeseries_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the numbers-parser library
!pip install numbers-parser

import requests
from numbers_parser import Document
import pandas as pd

# Raw GitHub URL for your file
url = "https://raw.githubusercontent.com/Se886372/Starbucks-Order-Interest-Over-Time/main/time_series_US_20031231-1900_20260710-1049.numbers"

# Download it into the Colab environment
file_path = "/content/time_series_US_20031231-1900_20260710-1049.numbers"
response = requests.get(url)
response.raise_for_status()  # errors out loudly if the fetch fails (e.g. 404)

with open(file_path, "wb") as f:
    f.write(response.content)

print(f"Downloaded {len(response.content):,} bytes to {file_path}")

# Load the document
doc = Document(file_path)

# A .numbers file can contain multiple sheets, each with multiple tables
# List what's available first so you can pick the right one
for sheet in doc.sheets:
    print(f"Sheet: {sheet.name}")
    for table in sheet.tables:
        print(f"  Table: {table.name} — {table.num_rows} rows x {table.num_cols} cols")

Downloaded 249,887 bytes to /content/time_series_US_20031231-1900_20260710-1049.numbers
Sheet: Sheet 1
  Table: time_series_US_20031231-1900_20260710-1049 — 272 rows x 9 cols


In [2]:
# Grab the first sheet and first table (adjust if your data lives elsewhere)
sheet = doc.sheets[0]
table = sheet.tables[0]

# Pull all rows, use the first row as the header
data = table.rows(values_only=True)
df = pd.DataFrame(data[1:], columns=data[0])

# Quick check
print(df.shape)
print(df.columns.tolist())
df.head(10)

(271, 9)
['Time', 'Pumpkin spice latte', 'Chai Latte', 'pink drink', 'Caramel Macchiato', 'nitro cold brew coffee', 'refresher', 'matcha tea', 'Frappuccino']


,Time,Pumpkin spice latte,Chai Latte,pink drink,Caramel Macchiato,nitro cold brew coffee,refresher,matcha tea,Frappuccino
0,2004-01-01,0.0,4.0,0.0,0.0,0.0,12.0,2.0,0.0
1,2004-02-01,0.0,2.0,0.0,0.0,0.0,10.0,0.0,0.0
2,2004-03-01,0.0,3.0,0.0,0.0,0.0,9.0,0.0,0.0
3,2004-04-01,0.0,2.0,0.0,0.0,0.0,10.0,0.0,0.0
4,2004-05-01,0.0,2.0,0.0,0.0,0.0,10.0,0.0,0.0
5,2004-06-01,0.0,1.0,2.0,0.0,0.0,11.0,0.0,0.0
6,2004-07-01,0.0,2.0,0.0,0.0,0.0,12.0,0.0,0.0
7,2004-08-01,0.0,2.0,1.0,0.0,0.0,10.0,0.0,0.0
8,2004-09-01,0.0,1.0,1.0,0.0,0.0,10.0,0.0,0.0
9,2004-10-01,2.0,2.0,1.0,0.0,0.0,9.0,0.0,0.0


In [3]:
import pandas as pd
import plotly.graph_objects as go

# --- Date column (already confirmed as "Time") ---
date_col = "Time"
df[date_col] = pd.to_datetime(df[date_col])
df = df.sort_values(date_col)

value_cols = ['Pumpkin spice latte', 'Chai Latte', 'pink drink', 'Caramel Macchiato',
              'nitro cold brew coffee', 'refresher', 'matcha tea', 'Frappuccino']

# --- 8 visually distinct colors (no repeats, no near-duplicate greens) ---
colors = [
    '#00704A',  # Starbucks green - Pumpkin spice latte
    '#C08552',  # warm tan - Chai Latte
    '#E472E0',  # orange - pink drink (contrast, not literal pink to stay readable on white) - CORRECTED FROM '#E472E'
    '#7B4B94',  # purple - Caramel Macchiato
    '#1B7A93',  # teal blue - nitro cold brew coffee
    '#D62839',  # red - refresher
    '#8AA624',  # olive/matcha green - matcha tea
    '#27251F',  # near-black - Frappuccino
]

fig = go.Figure()

for i, col in enumerate(value_cols):
    fig.add_trace(go.Scatter(
        x=df[date_col],
        y=df[col],
        mode='lines',
        name=col,
        line=dict(color=colors[i], width=3.0), # Increased line width to 3.0
        opacity=0.9,
        hovertemplate='<b>%{x|%b %d, %Y}</b><br>' + col + ': %{y}<extra></extra>'
    ))

fig.update_layout(
    title=dict(
        text="Starbucks Orders Search Interest Over Time",
        font=dict(size=22, family="Arial", color="#1E3932"),
        x=0.02
    ),
    xaxis=dict(
        title="Date",
        showgrid=False,
        rangeslider=dict(visible=True),
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1M", step="month", stepmode="backward"),
                dict(count=6, label="6M", step="month", stepmode="backward"),
                dict(count=1, label="1Y", step="year", stepmode="backward"),
                dict(count=5, label="5Y", step="year", stepmode="backward"),
                dict(step="all", label="All")
            ]),
            bgcolor="#F0F0F0"
        )
    ),
    yaxis=dict(
        title="Search Interest",
        showgrid=True,
        gridcolor='rgba(0,0,0,0.06)'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family="Arial", size=13, color="#333333"),
    hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=800, # Increased height to 800
    margin=dict(l=60, r=40, t=110, b=60)
)

fig.show()